# Capital Bikeshare (Washington, D.C.) — Starter Notebook

Hourly counts of bike rentals from the Capital Bikeshare system in 2011, with the weather and calendar for each hour (Fanaee-T & Gama, 2014; this version is from the *ISLP* textbook). Each row is **one hour**, not one person: the outcome is how many people rented a bike in that hour.

**How this notebook is organized**

| Section | What it does |
|---|---|
| Settings | Where the data lives (the only cell you might edit) |
| Load Data | Imports libraries, loads and prepares the data |
| Inspect the Dataset | First rows, size, column types |
| Describe Variables | Summary statistics |
| Check Missing Data | What's missing and why it matters |
| Basic Visualizations | A few plots to get oriented |
| Variables You May Want to Explore | Outcomes, predictors, grouping and time variables |
| Your Analysis Starts Here | Your space for HW 1, HW 2, and HW 3 |

Cells marked 🧰 are **starter code** written by the instructor. Cells marked ✏️ are **yours**.

Open the table of contents (the ☰ icon on the left in Colab) to jump between sections.

## Settings

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

If the default link stops working, or you downloaded the file yourself, change the path below. See the dataset guide for where to get the data.

In [ ]:
# Settings: where the data lives.
# By default the data load from this course's GitHub repository, so there's nothing to download.
REPO = "cogs118d/datasets"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/04_bikeshare/"

# Cloned the repo and working on your own computer? Use the local folder instead:
# BASE_URL = "../data/04_bikeshare/"
# Saved the files to Google Drive? Point BASE_URL at that folder, e.g.:
# BASE_URL = "/content/drive/MyDrive/04_bikeshare/"
DATA_URL = BASE_URL + "Bikeshare.csv"

# Load Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Libraries used in every COGS 118D starter notebook
import numpy as np                        # numbers and arrays
import pandas as pd                       # data tables ("DataFrames")
import matplotlib.pyplot as plt           # plots
import statsmodels.formula.api as smf     # GLMs and mixed models (you'll use this later)

pd.set_option("display.max_columns", 60)  # show more columns when printing tables
plt.rcParams["figure.figsize"] = (7, 4)   # default plot size

In [ ]:
df = pd.read_csv(DATA_URL)
print("Loaded", df.shape[0], "hours")

## Preprocessing

Months are stored as text; we put them in calendar order so plots come out right.

In [ ]:
month_order = ["Jan", "Feb", "March", "April", "May", "June", "July", "Aug", "Sept", "Oct", "Nov", "Dec"]
df["mnth"] = pd.Categorical(df["mnth"], categories=month_order, ordered=True)

# temp, atemp, hum, and windspeed are rescaled to 0-1 in this version (e.g., temp = Celsius / 41).
df["temp_celsius"] = df["temp"] * 41
df[["mnth", "day", "hr", "workingday", "weathersit", "temp_celsius", "bikers"]].head()

# Inspect the Dataset

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# The first 5 rows: each row is one observation
df.head()

In [ ]:
# Dimensions: (number of rows, number of columns)
print("Rows, columns:", df.shape)

In [ ]:
# Column names and data types (numbers, text, dates, True/False)
df.info()

# Describe Variables

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
df[["bikers", "casual", "registered", "temp_celsius", "hum", "windspeed"]].describe().round(2)

In [ ]:
print(df["weathersit"].value_counts(), "\n")
print("Working days vs. not (hours):"); print(df["workingday"].value_counts())

# Check Missing Data

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

In [ ]:
# Count and percent of missing values in each column (only columns with any missing)
missing = df.isna().sum()
missing_table = pd.DataFrame({"n_missing": missing, "percent": (100 * missing / len(df)).round(1)})
missing_table[missing_table["n_missing"] > 0].sort_values("n_missing", ascending=False)

**What to notice.** Nothing is missing, but a few hours of the year are absent (8,645 rows instead of 8,760), probably when the system recorded no rentals or was offline.

# Basic Visualizations

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

These plots are for getting oriented. They are not answers to any homework question.

In [ ]:
# Distribution of hourly rentals (a count with a long right tail)
plt.hist(df["bikers"], bins=50, edgecolor="white")
plt.xlabel("Rentals in the hour")
plt.ylabel("Number of hours")
plt.title("Hourly rentals")
plt.show()

In [ ]:
# Average rentals by hour of day, separately for working and non-working days
by_hour = df.groupby(["hr", "workingday"])["bikers"].mean().unstack()
plt.plot(by_hour.index, by_hour[1], marker=".", label="working day")
plt.plot(by_hour.index, by_hour[0], marker=".", label="weekend / holiday")
plt.xlabel("Hour of day")
plt.ylabel("Mean rentals")
plt.legend()
plt.title("Daily rhythm")
plt.show()

In [ ]:
# Rentals vs. temperature
plt.scatter(df["temp_celsius"], df["bikers"], s=3, alpha=0.3)
plt.xlabel("Temperature (°C)")
plt.ylabel("Rentals in the hour")
plt.title("Temperature and rentals")
plt.show()

In [ ]:
# Average rentals by weather and by month
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby("weathersit")["bikers"].mean().plot(kind="bar", ax=axes[0], title="By weather")
df.groupby("mnth", observed=True)["bikers"].mean().plot(kind="bar", ax=axes[1], title="By month")
axes[0].set_ylabel("Mean rentals per hour")
plt.tight_layout()
plt.show()

# Variables You May Want to Explore

> 🧰 **Starter code (provided).** Run these cells in order. You don't need to change anything except the **Settings** cell.

**Outcome variables:** `bikers` (total count), `casual` and `registered` (counts by rider type).

**Predictors:** `hr`, `workingday`, `holiday`, `weekday`, `mnth`, `season`, `weathersit`, `temp`, `atemp`, `hum`, `windspeed`.

Counts like these are usually far more spread out than a Poisson model allows, so plan to check for overdispersion.

In [ ]:
outcome_vars = ["bikers", "casual", "registered"]
predictor_vars = ["hr", "workingday", "temp", "hum", "windspeed"]
df[outcome_vars + predictor_vars].describe().round(2)

## Grouping structure (for mixed-effects models in HW 3)

Hours are grouped within **days** (`day` is the day of the year, 1–365) and days within **months**. Hours from the same day share weather and events, so they aren't independent.

In [ ]:
hours_per_day = df.groupby("day").size()
print("Days:", len(hours_per_day), "| hours per day (median):", hours_per_day.median())

## Time-to-event variables (for survival analysis in HW 3)

**Not appropriate for this dataset.** These are counts per hour, not times until an event. For HW 3, use a **mixed-effects model** with hours grouped within days.

---
# Your Analysis Starts Here

> ✏️ **Your analysis.** Everything below this line is yours. Add as many cells as you need.

The dataset guide lists starter questions for each homework. 

## HW 1 — Describe

In [ ]:
# ✏️ Your HW 1 code here


## HW 2 — Predict / Moderate / Mediate

In [ ]:
# ✏️ Your HW 2 code here


## HW 3 — Mixed Effects / Survival

In [ ]:
# ✏️ Your HW 3 code here
